# NorthForge Finance — End-to-End Workflow Run

Runs the full business workflow through `WorkflowOrchestrator`: Foundry's Trial Balance
pipeline (staging → enrichment → reporting → posting → interface) followed by the GL
import of that pipeline's Interface output — all under a single `WorkflowRun`. Recon is
then run explicitly against that same `workflow_run_id`, comparing Interface and GL.

Each section below reads and displays the data actually persisted at that stage, straight
from the domain repositories (`TrialBalanceRepository` for Foundry, `GLRepository` for
GL, `ReconClient` for recon).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

## Spark session

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/31 20:57:00 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/31 20:57:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fa81b81f-d97e-4177-aad7-ff7cd0db44fe;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 73ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

## Imports and helpers

In [3]:
from datetime import date

from pyspark.sql import functions as F

from core.logging import configure_logging
from foundry.pipeline import TrialBalancePipeline
from foundry.repository import TrialBalanceRepository
from foundry.config.settings import (
    CSV_TABLE_LOCATIONS,
    POSTGRES_TABLE_LOCATIONS
)

from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from spec import SpecClient
from atlas import AtlasClient
from reference import ReferenceClient

from registry import RegistryClient
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from workflow import WorkflowOrchestrator


configure_logging()


def display_df(df):
    display(df.toPandas())

## Configure clients and build the orchestrator

`TrialBalancePipeline` only knows Foundry processing now — no `RunTracker` —
`GLClient` only knows GL processing, and `ReconClient` only knows recon processing.
`WorkflowOrchestrator` is the thin layer that owns execution/workflow lifecycle across
Foundry and GL and coordinates Foundry → GL. Recon is not yet wired into the
orchestrator (v1 keeps it a standalone `recon.reconcile(workflow_run_id)` call,
run explicitly below once GL has posted).

In [4]:
BUSINESS_DT = date(2026, 3, 31)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

spec = SpecClient.from_db(
    spark,
    transformation_table = 'spec.transformation',
    file_layout_table = 'spec.file_layout'
    
)
atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)
reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

pipeline = TrialBalancePipeline(
    business_dt=BUSINESS_DT,
    repository=repository,
    spec=spec,
    atlas=atlas,
    reference=reference,
)

registry = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)
gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

orchestrator = WorkflowOrchestrator(
    run_tracker=run_tracker,
    foundry_pipeline=pipeline,
    gl=gl,
)

## Run the full workflow (Foundry → GL)

`run_workflow()` creates a single `WorkflowRun`, executes the Foundry pipeline
(`STAGING → ENRICHMENT → REPORTING → POSTING → INTERFACE`), then runs `GL / IMPORT`
against that same workflow's `FOUNDRY / INTERFACE` output — all as one business workflow.

In [ ]:
workflow_result = orchestrator.run_workflow()

business_dt = pipeline.config.business_dt

# V1 has one producer execution per workflow per output table, so
# workflow_run_id alone is the operational key for every read below.
# gl_run_id (GL's own producer_run_id) is kept only as exact lineage.
workflow_run_id = workflow_result.foundry.identity.workflow_run_id
gl_run_id = workflow_result.gl.producer_run_id

2026-08-31 20:57:03,398 | INFO | workflow.orchestrator | Workflow started | workflow_run_id=65dead20-f929-449e-88d6-fa5825ed372e | dataclass=TRIAL_BALANCE | business_dt=2026-03-31
2026-08-31 20:57:03,405 | INFO | workflow.orchestrator | Foundry pipeline started | run_id=833724d9-72d5-421f-8a81-5c45f9238b4b | workflow_run_id=65dead20-f929-449e-88d6-fa5825ed372e
2026-08-31 20:57:03,407 | INFO | workflow.orchestrator | Foundry zone started | operation=STAGING | run_id=f8e26504-a358-4e3d-b7bd-4211eb5ac798 | workflow_run_id=65dead20-f929-449e-88d6-fa5825ed372e
2026-08-31 20:57:09,150 | INFO | workflow.orchestrator | Foundry zone succeeded | operation=STAGING | run_id=f8e26504-a358-4e3d-b7bd-4211eb5ac798 | workflow_run_id=65dead20-f929-449e-88d6-fa5825ed372e | records=42
2026-08-31 20:57:09,156 | INFO | workflow.orchestrator | Foundry zone started | operation=ENRICHMENT | run_id=1d8ffeee-8636-45c5-82ae-e64db18bf2c9 | workflow_run_id=65dead20-f929-449e-88d6-fa5825ed372e


## Foundry persistence layers

Each Foundry zone is read straight from its own persisted table, selected by the
`workflow_run_id` of the workflow that produced it (via `TrialBalanceRepository`) — not
by `business_dt`/`batch_id`. `PRODUCER_RUN_ID` remains stamped on every row as the exact
producer execution's lineage.

### Source

In [ ]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

### Staging

In [ ]:
stg_df = repository.read_staging(workflow_run_id)

display_df(stg_df)

### Enrichment

In [ ]:
enr_df = repository.read_enrichment(workflow_run_id)

display_df(enr_df)

### Reporting

In [ ]:
rpt_df = repository.read_reporting(workflow_run_id)

display_df(rpt_df)

### Posting

In [ ]:
pst_df = repository.read_posting(workflow_run_id)

display_df(pst_df)

### Interface

This is the Foundry Interface output — the same rows GL reads as input for `GL / IMPORT`,
selected by `WORKFLOW_RUN_ID` rather than business date/batch.

In [ ]:
int_df = repository.read_interface(workflow_run_id)

display_df(int_df)

## GL persistence layer

`GL / IMPORT` (`gl_run_id`) reads the Interface rows produced by `FOUNDRY / INTERFACE`
above — selected by `workflow_run_id`, since V1 has one Interface producer execution per
workflow — and stamps its own execution identity onto whatever it writes: `gl.posting`
and `gl.rejection` rows carry `PRODUCER_RUN_ID = gl_run_id`, not the Interface producer's
ID. The reads below select by `workflow_run_id` too, not `gl_run_id`.

### GL Posting

In [ ]:
gl_postings = gl.get_postings(workflow_run_id)

display_df(gl_postings)

### GL Rejection

A non-zero rejected count here is a normal business outcome, not an execution failure —
`GL / IMPORT` still completes as `SUCCEEDED` as long as processing itself ran cleanly.

In [ ]:
gl_rejections = gl.get_rejections(workflow_run_id)

display_df(gl_rejections)

## Recon

`recon.reconcile(workflow_run_id)` reads `interface.trial_balance` and `gl.posting` for
the workflow — the GL side through the real `GLClient` abstraction, not a separate query
— aggregates each side to a balance per `RECON_KEYS` grain (`WORKFLOW_RUN_ID`,
`AS_OF_DATE`, and the nine GL segments plus `ACCOUNTED_CURRENCY`), and persists the
comparison to `recon.result`.

This runs under its own `RECON / RECONCILE` execution, created under the same
`workflow_run_id` being reconciled. That execution's own `run_id` is stamped as
`PRODUCER_RUN_ID` on every row it writes — distinct from `gl_run_id` above, which is GL's
own producer lineage for the postings being compared.

In [ ]:
recon_result = recon.reconcile(workflow_run_id)

recon_run_id = recon_result.producer_run_id

print(
    f"result_count={recon_result.result_count} "
    f"break_count={recon_result.break_count} "
    f"producer_run_id={recon_run_id}"
)

In [ ]:
recon_df = recon.get_results(workflow_run_id)

display_df(recon_df)

### Breaks

Rows with a non-zero `DIFFERENCE_AMOUNT` — expected to be empty for this balanced
synthetic run.

In [ ]:
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

display_df(breaks_df)

In [ ]:
spark.stop()